In [ ]:
import pandas as pd
import numpy as np
import kagglehub
import os
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OrdinalEncoder
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.impute import SimpleImputer

In [ ]:
path = kagglehub.dataset_download("maqdadali/mental-health-risk-prediction-dataset-students")

df = pd.read_csv(os.path.join(path, "student_mental_health.csv"))
print("Dataset columns:", df.columns)

Dataset columns: Index(['Student_ID', 'Age', 'Gender', 'Year_of_Study', 'Sleep_Hours',
       'Study_Hours_Per_Week', 'GPA', 'Social_Media_Hours',
       'Screen_Time_Hours', 'Physical_Activity', 'Financial_Stress',
       'Family_Support', 'Attendance_Rate', 'Part_Time_Job', 'Diet_Quality',
       'Previous_Mental_Health_History', 'Mental_Health_Risk'],
      dtype='str')


In [ ]:
features = ['Year_of_Study', 'Sleep_Hours', 'Study_Hours_Per_Week', 'GPA', 'Social_Media_Hours', 'Screen_Time_Hours', 'Financial_Stress',
       'Family_Support', 'Attendance_Rate', 'Physical_Activity_Low', 'Physical_Activity_Moderate',
       'Part_Time_Job_Yes', 'Diet_Quality_Good', 'Diet_Quality_Poor', 'Previous_Mental_Health_History_Yes']
target = ["Mental_Health_Risk"]

encoder = OrdinalEncoder(categories=[['Low', 'Medium', 'High']])

df['Mental_Health_Risk'] = encoder.fit_transform(df[['Mental_Health_Risk']]).ravel()

df = pd.get_dummies(df, drop_first=True)

correlation_matrix = df.corr()
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f')
plt.show()

print(df.shape)
print(df.head())
df.info()
print(df.describe())

print(df.duplicated().sum())
print(df.isnull().sum())

numerical_cols = df.select_dtypes(["float64", "int64"]).columns
categorical_cols = df.select_dtypes(["object", "category"]).columns

print(df.nunique().sum())

print(df[target].value_counts())
print(df[target].value_counts(normalize=True) * 100)

for col in numerical_cols:
    plt.figure(figsize=(6, 3))
    sns.boxplot(data=df, x=col)
    plt.title(f"Boxplot de {col}")
    plt.tight_layout()
    plt.show()

df = df.drop(columns=['Student_ID', 'Gender'])

print(df['Mental_Health_Risk'].value_counts())

sns.countplot(x='Mental_Health_Risk', data=df)
plt.show()

In [ ]:
X = df[features]
y = df['Mental_Health_Risk']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = LogisticRegression(random_state=42)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

print("Train accuracy:", model.score(X_train, y_train))
print("Test accuracy:", model.score(X_test, y_test))

ValueError: Input X contains NaN.
LogisticRegression does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

In [ ]:
imputer = SimpleImputer(strategy='mean')
X_train = imputer.fit_transform(X_train)
X_test = imputer.transform(X_test)

print("Train accuracy after imputation:", model.score(X_train, y_train))
print("Test accuracy after imputation:", model.score(X_test, y_test))

Train accuracy after imputation: 0.7344236760124611
Test accuracy after imputation: 0.7142857142857143


c:\Users\aymen\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
c:\Users\aymen\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(


In [ ]:
preprocessing = ColumnTransformer([
    ("scaler", MinMaxScaler(), X),
    ("imputer", SimpleImputer(strategy="median"), X)
])

preprocessingNumerical = ColumnTransformer([
    ("std", StandardScaler(), numerical_cols),
])

preprocessingCat = ColumnTransformer([
    ("oneh", OneHotEncoder(), categorical_cols),
])

pipeline = Pipeline([
    ("preprocessing", preprocessing),
    ("preprocessingNumerical", preprocessingNumerical),
    ("preprocessingCat", preprocessingCat),
    ("model", SVC())
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model.fit(X_train, y_train)

print("Train accuracy:", model.score(X_train, y_train))
print("Test accuracy:", model.score(X_test, y_test))

Train accuracy: 0.7468847352024922
Test accuracy: 0.7204968944099379


c:\Users\aymen\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
